# Eye 01

Eye-tracking / pupil analysis (part 1).

**Reads:** `data/individual/ (one participant, per session)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/individual/processed'

BLINK_THRESHOLD = 1.0
MIN_CLOSED_FRAMES = 3  # ~100ms at 30Hz

columns_mapping = {
    'datetime': 'timestamp',
    'pupil': 'pupil_dilation',
    'leftEyeOpen': 'left_blink',
    'rightEyeOpen': 'right_blink'
}

def clean_eye_tracking_data(eye_tracking_data):
    df = eye_tracking_data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce').dt.tz_convert(None)
    return df.dropna(subset=['timestamp'])

def count_blinks(series, threshold, min_frames):
    # sustained closure onset
    closed = (series <= threshold).astype(int)
    sustained = closed.rolling(min_frames).sum() == min_frames
    return int((sustained & ~sustained.shift(1, fill_value=False)).sum())

def calculate_eye_tracking_metrics(eye_tracking_data):
    start_time = eye_tracking_data['timestamp'].min()
    end_time = eye_tracking_data['timestamp'].max()
    total_duration_minutes = (end_time - start_time).total_seconds() / 60
    if total_duration_minutes <= 0:
        return start_time, end_time, total_duration_minutes, float('nan'), 0.0, 0.0

    average_pupil_dilation = eye_tracking_data['pupil_dilation'].mean()

    left_blink_rate = count_blinks(eye_tracking_data['left_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES) / total_duration_minutes
    right_blink_rate = count_blinks(eye_tracking_data['right_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES) / total_duration_minutes

    return start_time, end_time, total_duration_minutes, average_pupil_dilation, left_blink_rate, right_blink_rate

eye_tracking_baseline = clean_eye_tracking_data(pd.read_csv(f'{DATA}/sed.csv').rename(columns=columns_mapping))
eye_tracking_01 = clean_eye_tracking_data(pd.read_csv(f'{DATA}/sed_01.csv').rename(columns=columns_mapping))
eye_tracking_02 = clean_eye_tracking_data(pd.read_csv(f'{DATA}/sed_02.csv').rename(columns=columns_mapping))
eye_tracking_03 = clean_eye_tracking_data(pd.read_csv(f'{DATA}/sed_03.csv').rename(columns=columns_mapping))

baseline_metrics = calculate_eye_tracking_metrics(eye_tracking_baseline)
metrics_01 = calculate_eye_tracking_metrics(eye_tracking_01)
metrics_02 = calculate_eye_tracking_metrics(eye_tracking_02)
metrics_03 = calculate_eye_tracking_metrics(eye_tracking_03)

results = pd.DataFrame({
    'Test': ['Baseline', 'Test 01', 'Test 02', 'Test 03'],
    'Start Time': [baseline_metrics[0], metrics_01[0], metrics_02[0], metrics_03[0]],
    'End Time': [baseline_metrics[1], metrics_01[1], metrics_02[1], metrics_03[1]],
    'Total Duration (min)': [baseline_metrics[2], metrics_01[2], metrics_02[2], metrics_03[2]],
    'Average Pupil Dilation': [baseline_metrics[3], metrics_01[3], metrics_02[3], metrics_03[3]],
    'Left Blink Rate (blinks/min)': [baseline_metrics[4], metrics_01[4], metrics_02[4], metrics_03[4]],
    'Right Blink Rate (blinks/min)': [baseline_metrics[5], metrics_01[5], metrics_02[5], metrics_03[5]]
})

print("Eye-Tracking Metrics:")
for _, row in results.iterrows():
    print(f"{row['Test']} Metrics:")
    print(f"  Start Time: {row['Start Time']}")
    print(f"  End Time: {row['End Time']}")
    print(f"  Total Duration: {row['Total Duration (min)']:.2f} minutes")
    print(f"  Average Pupil Dilation: {row['Average Pupil Dilation']:.2f}")
    print(f"  Left Blink Rate: {row['Left Blink Rate (blinks/min)']:.2f} blinks/min")
    print(f"  Right Blink Rate: {row['Right Blink Rate (blinks/min)']:.2f} blinks/min\n")

plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(results['Test'], results['Average Pupil Dilation'], marker='o', linestyle='-')
plt.title('Average Pupil Dilation')
plt.ylabel('Pupil Dilation')

plt.subplot(3, 1, 2)
plt.plot(results['Test'], results['Left Blink Rate (blinks/min)'], marker='o', linestyle='-', color='green')
plt.title('Left Blink Rate')
plt.ylabel('Blinks/min')

plt.subplot(3, 1, 3)
plt.plot(results['Test'], results['Right Blink Rate (blinks/min)'], marker='o', linestyle='-', color='red')
plt.title('Right Blink Rate')
plt.ylabel('Blinks/min')

plt.tight_layout()
plt.show()
plt.close()
